# 02 - TIỀN XỬ LÝ - PlantVillage

## Mục tiêu

Tạo **bộ dữ liệu đầu vào nhất quán** để `03_simple_cnn.ipynb`, `04_complex_cnn.ipynb` và `05_transfer_learning.ipynb` sử dụng chung.

1. Đọc `metadata.csv` và các danh sách vấn đề đã tạo trong bước EDA.
2. Loại ảnh lỗi khỏi **danh sách ảnh sử dụng** mà không xóa file gốc.
3. Kiểm tra nhãn mâu thuẫn trong các nhóm ảnh trùng và gần trùng.
4. Loại các bản sao trùng hoàn toàn khỏi danh sách ảnh sử dụng.
5. Gom mọi cặp cùng lớp do pHash phát hiện vào cùng `group_id`; cặp khác lớp chỉ gom sau khi đạt ngưỡng pixel. Cách này tránh để ảnh chụp cùng một lá ở nhiều tập khi phép đo pixel chỉ thiếu ngưỡng một chút.
6. Dùng các ảnh `val` gốc còn lại sau làm sạch làm **test**; nếu một thành viên của nhóm thuộc `val` gốc thì đưa cả nhóm vào test.
7. Chia phần còn lại của `train` gốc thành **87,5% train / 12,5% validation**, có xét lớp và nhóm ảnh.
8. Cố định `SEED = 42` và lưu `data_split.csv` để mọi mô hình dùng cùng một bộ chia.
9. Tạo quy trình đọc ảnh theo batch, chuyển sang RGB và đổi kích thước thành `224 x 224`.
10. Chỉ tăng cường dữ liệu cho tập train; mô hình pretrained dùng phép chuẩn hóa theo bộ trọng số đã chọn.
11. Lưu cấu hình, thứ tự lớp và bộ chia dữ liệu.

### Tỷ lệ mục tiêu trước khi làm sạch

| Tập | Nguồn | Số ảnh trước khi làm sạch |
|---|---|---:|
| Train | 87,5% của `train` gốc | ~38.013 |
| Validation | 12,5% của `train` gốc | ~5.431 |
| Test | toàn bộ `val` gốc | 10.861 |

Ảnh lỗi, ảnh trùng và nhóm ảnh gần trùng có thể làm thay đổi cách chia, nên **số lượng thực tế sau khi làm sạch có thể khác bảng trên**.

Các bước xử lý dùng hàm chung trong `src/data_utils.py`; notebook này hiển thị kết quả và nhận xét.


# I. THƯ VIỆN VÀ CẤU HÌNH


In [ ]:
from pathlib import Path
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
N_SPLITS = 8
CROSS_CLASS_PHASH_THRESHOLD = 5
CROSS_CLASS_MAX_PIXEL_MAE = 0.08
CROSS_CLASS_MIN_PIXEL_CORRELATION = 0.90


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Phiên bản PyTorch:", torch.__version__)
print("Thiết bị:", DEVICE)


# II. ĐƯỜNG DẪN DỰ ÁN, DỮ LIỆU VÀ ĐẦU RA


In [ ]:
def find_project_dir():
    override = os.getenv("PLANT_DISEASE_PROJECT_DIR")
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "src" / "data_utils.py").is_file():
            return candidate
        raise FileNotFoundError(f"Thư mục dự án không hợp lệ: {candidate}")

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, cwd / "plant-disease-classification"]
    for candidate in candidates:
        if (candidate / "src" / "data_utils.py").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy dự án. Hãy đặt PLANT_DISEASE_PROJECT_DIR.")

PROJECT_DIR = find_project_dir()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

sys.dont_write_bytecode = True

from src.data_utils import (
    load_preprocessing_inputs,
    find_broken_and_missing_images,
    find_cross_class_near_duplicates,
    build_duplicate_groups,
    clean_metadata,
    build_class_mapping,
    build_data_split,
    SPLIT_COLUMNS,
    validate_split,
    make_transforms,
    make_transfer_transforms,
    make_dataloaders,
    save_preprocessing_outputs,
)

load_dotenv(PROJECT_DIR / ".env")

data_dir_value = os.getenv("DATA_DIR")
DATA_DIR = Path(data_dir_value) if data_dir_value else PROJECT_DIR.parent / "PlantVillage"
if not DATA_DIR.is_absolute():
    DATA_DIR = PROJECT_DIR / DATA_DIR
DATA_DIR = DATA_DIR.resolve()

METADATA_DIR = PROJECT_DIR / "data" / "metadata"
RESULTS_DIR = PROJECT_DIR / "outputs" / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = METADATA_DIR / "metadata.csv"
DUPLICATE_PATH = METADATA_DIR / "duplicate_images.csv"
NEAR_DUPLICATE_PATH = METADATA_DIR / "near_duplicates.csv"
BROKEN_PATH = METADATA_DIR / "broken_images.csv"

DATA_SPLIT_PATH = RESULTS_DIR / "data_split.csv"
CLASS_NAMES_PATH = RESULTS_DIR / "class_names.json"
CONFIG_PATH = RESULTS_DIR / "preprocessing_config.json"
CROSS_CLASS_REVIEW_PATH = RESULTS_DIR / "cross_class_near_duplicate_candidates.csv"

print("Thư mục dự án :", PROJECT_DIR)
print("Thư mục dữ liệu:", DATA_DIR)
print("Thư mục metadata:", METADATA_DIR)
print()
print("metadata.csv tồn tại       :", "Có" if METADATA_PATH.exists() else "Không")
print("duplicate_images.csv tồn tại:", "Có" if DUPLICATE_PATH.exists() else "Không")
print("near_duplicates.csv tồn tại :", "Có" if NEAR_DUPLICATE_PATH.exists() else "Không")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy DATA_DIR: {DATA_DIR}")

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy data/metadata/metadata.csv. "
        "Hãy chạy 01_eda.ipynb trước."
    )

if not DUPLICATE_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy duplicate_images.csv. "
        "Hãy chạy phần kiểm tra ảnh trùng trong 01_eda.ipynb trước."
    )

if not NEAR_DUPLICATE_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy near_duplicates.csv. "
        "Hãy chạy phần kiểm tra ảnh gần trùng trong 01_eda.ipynb trước."
    )


# III. ĐỌC METADATA VÀ DANH SÁCH VẤN ĐỀ TỪ EDA

Các file đầu vào, trong đó `broken_images.csv` là tùy chọn:

```text
data/metadata/
|-- metadata.csv
|-- duplicate_images.csv
|-- near_duplicates.csv
+-- broken_images.csv  (tùy chọn)
```

Dù EDA có lưu `broken_images.csv` hay không, notebook vẫn kiểm tra tính toàn vẹn và giải mã tất cả ảnh trước khi chia dữ liệu.


In [ ]:
metadata, duplicate_df, near_duplicate_df, broken_df = load_preprocessing_inputs(
    METADATA_PATH, DUPLICATE_PATH, NEAR_DUPLICATE_PATH, BROKEN_PATH
)

print("Số dòng metadata       :", len(metadata))
print("Số dòng ảnh trùng hẳn :", len(duplicate_df))
print("Số cặp ảnh gần trùng  :", len(near_duplicate_df))
print("Số dòng ảnh lỗi đã đọc:", len(broken_df))
display(metadata.head())


# IV. KIỂM TRA ẢNH LỖI VÀ FILE BỊ THIẾU


In [ ]:
print("Đang kiểm tra tính toàn vẹn và giải mã toàn bộ ảnh...")

broken_df, broken_paths, missing_file_paths = find_broken_and_missing_images(
    metadata,
    DATA_DIR,
    broken_df=broken_df,
    verify_images=True,
)

print("Ảnh lỗi:", len(broken_paths))
print("File bị thiếu:", len(missing_file_paths))
if len(broken_df) > 0:
    display(broken_df.head())


# V. GOM NHÓM ẢNH TRÙNG, GẦN TRÙNG VÀ KIỂM TRA NHÃN MÂU THUẪN

`near_duplicates.csv` từ EDA chứa các **cặp cùng lớp** có khoảng cách pHash tối đa 5. Mọi cặp này được gom nhóm để tránh ảnh gần trùng nằm ở nhiều tập. Bước preprocessing quét thêm cặp **khác lớp**; chỉ cặp đạt cả ngưỡng MAE và tương quan pixel mới được gom nhóm. Nhóm nhiều nhãn sẽ bị loại.

Báo cáo cặp khác lớp được lưu để xem xét thủ công. Cách phát hiện bằng pHash có thể bỏ sót ảnh gần trùng nằm ngoài ngưỡng.


In [ ]:
usable_for_hash = metadata[
    ~metadata["relative_path"].isin(broken_paths | missing_file_paths)
]
usable_paths = set(usable_for_hash["relative_path"])
usable_near_df = near_duplicate_df[
    near_duplicate_df["image_1"].isin(usable_paths)
    & near_duplicate_df["image_2"].isin(usable_paths)
].copy().reset_index(drop=True)
skipped_near_candidates = len(near_duplicate_df) - len(usable_near_df)

cross_class_candidates = find_cross_class_near_duplicates(
    usable_for_hash, DATA_DIR,
    threshold=CROSS_CLASS_PHASH_THRESHOLD,
    max_pixel_mae=CROSS_CLASS_MAX_PIXEL_MAE,
    min_pixel_correlation=CROSS_CLASS_MIN_PIXEL_CORRELATION,
)
cross_class_near_df = cross_class_candidates[
    cross_class_candidates["confirmed"]
].copy()
near_duplicate_df = pd.concat(
    [usable_near_df, cross_class_near_df], ignore_index=True
)
print("Cặp cùng lớp từ EDA được gom nhóm:", len(usable_near_df))
print("Cặp có ảnh lỗi hoặc thiếu đã bỏ qua:", skipped_near_candidates)
print("Cặp khác lớp theo pHash:", len(cross_class_candidates))
print("Cặp khác lớp đạt ngưỡng pixel:", len(cross_class_near_df))

metadata, conflict_groups, label_conflict_df = build_duplicate_groups(
    metadata, duplicate_df, near_duplicate_df,
    excluded_paths=broken_paths | missing_file_paths,
)

print("Tổng số nhóm:", metadata["group_id"].nunique())
print("Nhóm có nhiều nhãn:", len(conflict_groups))
print("Ảnh trong nhóm có nhãn mâu thuẫn:", len(label_conflict_df))
if len(label_conflict_df) > 0:
    display(label_conflict_df[
        ["relative_path", "class_name", "original_split", "group_id"]
    ].head(30))


# VI. LÀM SẠCH DANH SÁCH ẢNH

Quy tắc:

1. Loại ảnh lỗi và file bị thiếu.
2. Loại toàn bộ ảnh trong nhóm có nhãn mâu thuẫn.
3. Với **ảnh trùng hoàn toàn** (cùng mã băm SHA256), chỉ giữ một bản trong danh sách:
   - nếu nhóm có ảnh từ tập `val` gốc, giữ bản thuộc `val` để bảo toàn tập test;
   - nếu không, giữ bản có `relative_path` nhỏ nhất theo thứ tự từ điển.
4. Mặc định giữ ảnh gần trùng, nhưng gán chúng cùng `group_id`.

Không xóa file ảnh gốc nào khỏi ổ đĩa.


In [ ]:
clean_df, excluded_reasons, exact_duplicate_removed = clean_metadata(
    metadata, duplicate_df, broken_paths, missing_file_paths, label_conflict_df
)

print("Ảnh trong metadata ban đầu:", len(metadata))
print("Ảnh lỗi hoặc file thiếu  :", len(broken_paths | missing_file_paths))
print("Ảnh có nhãn mâu thuẫn   :", len(label_conflict_df))
print("Bản sao trùng đã loại   :", len(exact_duplicate_removed))
print("Ảnh còn sử dụng         :", len(clean_df))

reason_counts = {}
for reasons in excluded_reasons.values():
    for reason in reasons:
        reason_counts[reason] = reason_counts.get(reason, 0) + 1

print("\nChi tiết lý do loại ảnh:")
for reason, count in sorted(reason_counts.items()):
    print(f"  {reason}: {count}")


# VII. CHUẨN HÓA NHÃN

Cố định ánh xạ lớp bằng cách **sắp xếp tên lớp theo bảng chữ cái**.

`class_names.json` lưu đúng thứ tự này.  
Mã lớp là vị trí của lớp trong danh sách, từ `0` đến `37`.


In [ ]:
clean_df, class_names, class_to_idx, idx_to_class = build_class_mapping(
    metadata, clean_df
)
mapping_df = pd.DataFrame({
    "class_id": range(len(class_names)),
    "class_name": class_names,
})
print("Số lớp:", len(class_names))
display(mapping_df)


# VIII. CHIA TẬP TRAIN / VALIDATION / TEST

## Quy tắc chia

- **Test:** toàn bộ ảnh từ tập `val` gốc còn lại sau làm sạch.
- Nếu một `group_id` có ít nhất một ảnh từ `val` gốc, đưa **cả nhóm** vào test để tránh rò rỉ dữ liệu.
- Các ảnh còn lại phải đến từ tập `train` gốc.
- Chia phần `train` gốc còn lại thành:
  - `87,5%` train;
  - `12,5%` validation.
- Dùng `StratifiedGroupKFold(n_splits=8)`:
  - phân tầng theo `class_name`;
  - gom nhóm theo `group_id`;
  - `shuffle=True`;
  - `random_state=42`.

Một trong tám fold chiếm khoảng `12,5%`, phù hợp với tỷ lệ validation mục tiêu.


In [ ]:
split_input_df = clean_df.sort_values("relative_path").reset_index(drop=True)
split_df, selected_fold, test_group_count = build_data_split(
    split_input_df, seed=SEED, n_splits=N_SPLITS
)
split_df = split_df[SPLIT_COLUMNS].sort_values(
    ["split", "class_id", "relative_path"]
).reset_index(drop=True)

print("Đã tạo bộ chia từ dữ liệu đã xác nhận (sẽ lưu sau khi kiểm tra):",
      DATA_SPLIT_PATH)
print("Fold được chọn:", selected_fold)
print("Số nhóm test :", test_group_count)
display(split_df["split"].value_counts().rename("count").to_frame())


# IX. KIỂM TRA BỘ CHIA DỮ LIỆU


In [ ]:
validation_report = validate_split(
    split_df, near_duplicate_df, class_names
)
leaking_groups = validation_report["leaking_groups"]
near_pair_leaks = validation_report["near_pair_leaks"]
split_summary = validation_report["split_summary"]
class_split_counts = validation_report["class_split_counts"]

print("Không có nhóm hoặc cặp gần trùng cùng lớp nào bị tách giữa các tập.")
print()
display(split_summary)
display(class_split_counts)


In [ ]:
missing_class_rows = validation_report["missing_class_rows"]
if missing_class_rows:
    details = "; ".join(
        f'{item["split"]}: {item["missing_classes"]}'
        for item in missing_class_rows
    )
    raise ValueError(f"Có tập còn thiếu lớp: {details}")
print("OK: train, validation và test đều có đủ", len(class_names), "lớp.")


# X. QUY TRÌNH ĐỌC ẢNH

`data_split.csv` là nguồn duy nhất xác định ảnh thuộc train, validation hay test.

Quy trình tiền xử lý:

- chỉ đọc ảnh khi cần, không nạp toàn bộ dữ liệu vào RAM;
- chuyển ảnh sang `RGB`;
- đổi kích thước ảnh thành `224 x 224`;
- train: lật ngang, xoay nhẹ, phóng to hoặc thu nhỏ nhẹ và điều chỉnh tương phản nhẹ;
- chuyển ảnh thành tensor với giá trị pixel trong `[0, 1]`;
- validation/test: chỉ đổi kích thước và chuyển thành tensor, không tăng cường dữ liệu.

Các phép biến đổi trên dành cho CNN tự huấn luyện. Với mô hình pretrained, dùng `make_transfer_transforms(weights)` sau khi chọn bộ trọng số; hàm này áp dụng đúng phép chuẩn hóa từ `weights.transforms()`. Ví dụ:

```python
from torchvision.models import ResNet18_Weights
weights = ResNet18_Weights.DEFAULT
transfer_train_transform, transfer_eval_transform = make_transfer_transforms(weights)
transfer_loaders = make_dataloaders(
    split_df, DATA_DIR, transfer_train_transform, transfer_eval_transform,
    batch_size=BATCH_SIZE, num_workers=0,
)
```

Khi dùng bộ trọng số khác, thay `weights` cho đúng mô hình đã chọn.


In [ ]:
train_transform, eval_transform = make_transforms(image_size=IMG_SIZE)
print("Biến đổi tập train:")
print(train_transform)
print("\nBiến đổi tập validation/test:")
print(eval_transform)


In [ ]:
train_split_df = split_df[split_df["split"] == "train"].copy()
validation_split_df = split_df[split_df["split"] == "validation"].copy()
test_split_df = split_df[split_df["split"] == "test"].copy()

print("Số ảnh train      :", len(train_split_df))
print("Số ảnh validation :", len(validation_split_df))
print("Số ảnh test       :", len(test_split_df))


# XI. DATALOADER / BATCH

Chia dữ liệu thành các batch giúp sử dụng bộ nhớ vừa phải và cho mô hình học qua nhiều bước.


In [ ]:
# num_workers=0 hoạt động ổn định trong Jupyter/VS Code trên Windows.
train_loader, validation_loader, test_loader = make_dataloaders(
    split_df,
    DATA_DIR,
    train_transform,
    eval_transform,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

train_dataset = train_loader.dataset
validation_dataset = validation_loader.dataset
test_dataset = test_loader.dataset

print("Số batch train      :", len(train_loader))
print("Số batch validation :", len(validation_loader))
print("Số batch test       :", len(test_loader))


# XII. KIỂM TRA MỘT BATCH SAU TIỀN XỬ LÝ


In [ ]:
images, labels = next(
    iter(train_loader)
)

print("Kích thước batch ảnh:", images.shape)
print("Kích thước batch nhãn:", labels.shape)
print("Kiểu dữ liệu ảnh    :", images.dtype)
print("Kiểu dữ liệu nhãn   :", labels.dtype)
print(
    "Khoảng giá trị pixel:",
    float(images.min()),
    "->",
    float(images.max())
)
print(
    "Khoảng giá trị nhãn :",
    int(labels.min()),
    "->",
    int(labels.max())
)

assert images.ndim == 4
assert images.shape[1] == 3
assert images.shape[2] == IMG_SIZE
assert images.shape[3] == IMG_SIZE
assert float(images.min()) >= 0.0
assert float(images.max()) <= 1.0


# XIII. LƯU THỨ TỰ LỚP VÀ CẤU HÌNH


In [ ]:
split_count_dict = {
    name: int(
        (split_df["split"] == name).sum()
    )
    for name in [
        "train",
        "validation",
        "test"
    ]
}

config = {
    "seed": SEED,
    "image_size": [
        IMG_SIZE,
        IMG_SIZE
    ],
    "batch_size": BATCH_SIZE,
    "num_classes": len(class_names),
    "paths": {
        "metadata": str(
            METADATA_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/"),
        "duplicate_issues": str(
            DUPLICATE_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/"),
        "near_duplicate_issues": str(
            NEAR_DUPLICATE_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/"),
        "data_split": str(
            DATA_SPLIT_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/"),
        "class_names": str(
            CLASS_NAMES_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/"),
        "cross_class_candidate_report": str(
            CROSS_CLASS_REVIEW_PATH.relative_to(PROJECT_DIR)
        ).replace("\\", "/")
    },
    "cleaning": {
        "metadata_rows_before_cleaning": int(
            len(metadata)
        ),
        "broken_images": int(
            len(broken_paths)
        ),
        "missing_files": int(
            len(missing_file_paths)
        ),
        "label_conflict_groups": int(
            len(conflict_groups)
        ),
        "label_conflict_images": int(
            len(label_conflict_df)
        ),
        "exact_duplicate_copies_removed": int(
            len(exact_duplicate_removed)
        ),
        "same_class_candidates_grouped": int(len(usable_near_df)),
        "same_class_candidates_skipped": int(skipped_near_candidates),
        "cross_class_phash_candidates": int(
            len(cross_class_candidates)
        ),
        "cross_class_near_duplicates_confirmed": int(
            len(cross_class_near_df)
        ),
        "rows_after_cleaning": int(
            len(clean_df)
        )
    },
    "split_strategy": {
        "same_class_grouping": "Toàn bộ cặp cùng lớp do EDA tìm bằng pHash",
        "cross_class_phash_threshold": CROSS_CLASS_PHASH_THRESHOLD,
        "cross_class_max_pixel_mae": CROSS_CLASS_MAX_PIXEL_MAE,
        "cross_class_min_pixel_correlation": CROSS_CLASS_MIN_PIXEL_CORRELATION,
        "test": (
            "Toàn bộ ảnh val gốc còn dùng được; "
            "nếu nhóm ảnh trùng hoặc gần trùng có ảnh thuộc val gốc, "
            "đưa cả nhóm vào test."
        ),
        "train_validation_source": (
            "Ảnh train gốc còn dùng được và không thuộc nhóm đã đưa vào test."
        ),
        "validation_fraction_of_remaining_train": 0.125,
        "splitter": "StratifiedGroupKFold",
        "n_splits": N_SPLITS,
        "selected_fold": (
            None
            if selected_fold is None
            else int(selected_fold)
        ),
        "stratify_column": "class_name",
        "group_column": "group_id"
    },
    "final_split_counts": split_count_dict,
    "custom_cnn_preprocessing": {
        "color": "RGB",
        "resize": [
            IMG_SIZE,
            IMG_SIZE
        ],
        "pixel_range": [
            0.0,
            1.0
        ],
        "train_augmentation": {
            "horizontal_flip_probability": 0.5,
            "rotation_degrees": 10,
            "zoom_scale": [
                0.90,
                1.10
            ],
            "contrast_jitter": 0.10
        },
        "validation_augmentation": None,
        "test_augmentation": None
    },
    "transfer_learning_preprocessing": (
        "Dùng make_transfer_transforms(weights) với đúng bộ trọng số "
        "pretrained đã chọn; phép chuẩn hóa lấy từ weights.transforms()."
    )
}

cross_class_candidates.to_csv(CROSS_CLASS_REVIEW_PATH, index=False)

save_preprocessing_outputs(
    split_df,
    class_names,
    config,
    DATA_SPLIT_PATH,
    CLASS_NAMES_PATH,
    CONFIG_PATH,
    save_split=True,
)

print("Đã lưu bộ chia:", DATA_SPLIT_PATH)
print("Đã lưu:", CLASS_NAMES_PATH)
print("Đã lưu:", CONFIG_PATH)


# XIV. TỔNG KẾT


In [ ]:
print("=" * 65)
print("TỔNG KẾT TIỀN XỬ LÝ")
print("=" * 65)

print(
    f"Metadata trước khi làm sạch: {len(metadata):,}"
)
print(
    f"Dữ liệu sau khi làm sạch : {len(split_df):,}"
)
print(
    f"Số lớp                   : {len(class_names)}"
)

print()

for split_name in [
    "train",
    "validation",
    "test"
]:
    count = int(
        (split_df["split"] == split_name).sum()
    )

    ratio = (
        count / len(split_df)
        if len(split_df) > 0
        else 0
    )

    print(
        f"{split_name:<11}: "
        f"{count:>6,} "
        f"({ratio:.2%})"
    )

print()
print(
    f"Nhóm bị rò rỉ giữa các tập: {len(leaking_groups)}"
)
print(
    f"Cặp gần trùng bị tách tập: {len(near_pair_leaks)}"
)
print(
    "Kích thước ảnh           : "
    f"{IMG_SIZE} x {IMG_SIZE}"
)
print(
    "Dải pixel cho CNN tùy chỉnh: [0, 1]"
)
print(
    "Tăng cường dữ liệu train  : Có"
)
print(
    "Tăng cường validation/test : Không"
)

print("\nĐầu ra:")
print(" -", DATA_SPLIT_PATH)
print(" -", CLASS_NAMES_PATH)
print(" -", CONFIG_PATH)
print(" -", CROSS_CLASS_REVIEW_PATH)


## Đầu ra để các notebook mô hình sử dụng độc lập

Các notebook mô hình **không cần chạy lại notebook này** nếu các file đầu ra đã tồn tại.

### `outputs/results/data_split.csv`

Các cột chính:

```text
relative_path
class_name
class_id
split
original_split
group_id
```

### `outputs/results/class_names.json`

Danh sách tên lớp theo đúng thứ tự `class_id`.

### `outputs/results/preprocessing_config.json`

Lưu seed, kích thước ảnh, kích thước batch, tóm tắt bước làm sạch, cách chia dữ liệu và cấu hình tăng cường dữ liệu.

### `outputs/results/cross_class_near_duplicate_candidates.csv`

Danh sách cặp ảnh khác lớp gần nhau theo pHash, kèm độ giống pixel và cờ `confirmed`. Chỉ các cặp khác lớp đạt ngưỡng pixel mới ảnh hưởng đến việc gom nhóm; các cặp còn lại cần xem xét thủ công.

Ba notebook mô hình phải đọc lại `data_split.csv` và `class_names.json` thay vì tự chia dữ liệu lần nữa.
